# Comparing two groups: tests and effect sizes

Core stage, lesson C9. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** run a two-sample test for a proportion and for a mean, read its output, report an effect size and an interval next to the p-value, tell statistical from practical significance, and write the verdict in two sentences for a manager.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, GREEN, PURPLE, GOLD, RED = "#0072b2", "#00a07a", "#9a6be0", "#ad9100", "#c0392b"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, GREEN, PURPLE, GOLD]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
users = pd.read_csv(BASE + "users.csv")
assign = pd.read_csv(BASE + "experiment_assignments.csv")
events = pd.read_csv(BASE + "events.csv", parse_dates=["event_ts"], usecols=["user_id", "event_ts", "event_name", "platform"])    # 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people and 15,868 experiment rows.
assert (len(users), len(assign)) == (21841, 15868)

# The platform of a person is the platform of their signup_complete event; its time is the signup time.
signup = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")
users["platform"] = users["user_id"].map(signup["platform"])
users["signup_ts"] = users["user_id"].map(signup["event_ts"])

# Activity: the session_start and add_expense events of a person in the first 14 days after signing up (day 14 included).
use = events[events["event_name"].isin(["session_start", "add_expense"])].merge(users[["user_id", "signup_ts"]], on="user_id")
first14 = use[(use["event_ts"] >= use["signup_ts"]) & (use["event_ts"] <= use["signup_ts"] + pd.Timedelta(days=14))]
users["activity"] = users["user_id"].map(first14.groupby("user_id").size()).fillna(0)

# Bought: a purchase event within 14 days of signing up.
buys = events[events["event_name"] == "purchase"].merge(users[["user_id", "signup_ts"]], on="user_id")
buys = buys[buys["event_ts"] <= buys["signup_ts"] + pd.Timedelta(days=14)]
users["bought"] = users["user_id"].isin(buys["user_id"]).astype(int)

# People who signed up by June 16 have a full 14 days of follow-up before the data ends (June 30).
full = users[users["signup_date"] <= "2026-06-16"]
assert round(float(full["bought"].mean()), 3) == 0.12      # the README: 12.0 percent
print(f"Data loaded: {len(full):,} people with a full 14 days of follow-up, {100 * full['bought'].mean():.1f} percent bought Plus.")

## Unit 1: Two proportions

**Predict first.** In paywall_v2, 12.7 percent of the control group and 15.3 percent of the variant group bought Plus within 14 days. Will the p-value be above or below 0.05? Will the chi-square test of a table of counts agree with the z test? And will the 95 percent interval of the difference stay above 0, and does it reach 1 point?

In [ ]:
pw = assign[assign["experiment"] == "paywall_v2"]
ctrl = pw.loc[pw["variant"] == "control", "converted"]               # True = bought Plus within 14 days
var = pw.loc[pw["variant"] == "variant", "converted"]
n0, n1 = len(ctrl), len(var)                                         # people in control and in the variant
c0, c1 = int(ctrl.sum()), int(var.sum())                             # buyers
p0, p1 = c0 / n0, c1 / n1                                            # conversion rates
print(n0, n1, c0, c1, round(p0, 4), round(p1, 4))
pooled = (c0 + c1) / (n0 + n1)                                       # if the null is true, both groups share one rate
se = np.sqrt(pooled * (1 - pooled) * (1 / n0 + 1 / n1))              # the standard error of the gap under the null
z = (p1 - p0) / se
p_value = 2 * stats.norm.sf(abs(z))                                  # both tails of the bell curve
print(round(float(pooled), 4), round(float(se), 4), round(float(z), 2), round(float(p_value), 3))
table = [[c1, n1 - c1], [c0, n0 - c0]]                               # buyers and non-buyers in each group
chi2 = stats.chi2_contingency(table, correction=False)               # the same test, from a table of counts
print(round(float(chi2.statistic), 2), round(float(z ** 2), 2), round(float(chi2.pvalue), 3))
diff = p1 - p0
se_diff = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)           # the standard error of the difference, from the two groups
low, high = diff - 1.96 * se_diff, diff + 1.96 * se_diff
print(round(float(100 * diff), 2), round(float(100 * low), 2), round(float(100 * high), 2), round(float(100 * diff / p0), 1))

## Unit 2: Two means

**Predict first.** iOS people are more active than Android people in their first 14 days. Will Welch's test give a p-value above or below 0.001? Will the 95 percent interval of the difference of the means reach 0.5 events?

In [ ]:
ios = full.loc[full["platform"] == "ios", "activity"].to_numpy()
android = full.loc[full["platform"] == "android", "activity"].to_numpy()
print(len(ios), len(android), round(float(ios.mean()), 2), round(float(android.mean()), 2), round(float(ios.mean() - android.mean()), 2))
se_w = np.sqrt(ios.var(ddof=1) / len(ios) + android.var(ddof=1) / len(android))     # Welch: each group keeps its own variance
t_hand = (ios.mean() - android.mean()) / se_w
welch = stats.ttest_ind(ios, android, equal_var=False)
print(round(float(se_w), 3), round(float(t_hand), 2), round(float(welch.statistic), 2), f"{welch.pvalue:.1e}", round(float(welch.df)))
ci = welch.confidence_interval(confidence_level=0.95)               # the 95 percent interval of the difference of the means
print(round(float(ci.low), 2), round(float(ci.high), 2))
print(f"{stats.mannwhitneyu(ios, android).pvalue:.1e}")              # a test that does not use the means
print(round(float((ios == 0).mean()), 3), round(float((android == 0).mean()), 3))     # the share of people with no event at all

## Unit 3: The size of the gap, next to the p-value

**Predict first.** The cell prints the pooled SD of activity and Cohen's d for iOS against Android, then repeats the test with 50, 200, 800 and 3,200 people picked at random from each platform. As the groups shrink, what happens to the p-value, and what happens to d?

In [ ]:
def cohens_d(a, b):
    """The difference of the means in units of the pooled standard deviation."""
    pooled_sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd


sd_pooled = np.sqrt(((len(ios) - 1) * ios.var(ddof=1) + (len(android) - 1) * android.var(ddof=1)) / (len(ios) + len(android) - 2))
print(round(float(sd_pooled), 2), round(float(cohens_d(ios, android)), 2))
rng = np.random.default_rng(8)
for size in (50, 200, 800, 3200):                                    # the same two populations, only fewer people
    a = rng.choice(ios, size=size, replace=False)
    b = rng.choice(android, size=size, replace=False)
    print(size, f"{stats.ttest_ind(a, b, equal_var=False).pvalue:.2g}", round(float(cohens_d(a, b)), 2))

## Unit 4: The verdict for a manager

**Predict first.** Before you run the cell, write two sentences for the product manager about paywall_v2. What must the first one hold, and what must the second one do?

In [ ]:
verdict = (f"paywall_v2 raised 14-day conversion from {100 * p0:.1f}% to {100 * p1:.1f}%, {100 * diff:.1f} points more (95% interval {100 * low:.1f} to {100 * high:.1f} points, p = {p_value:.3f}). "
           f"If the team needs at least 1 point to keep the redesign, the data point that way, but the interval's low end, {100 * low:.1f}, is under that bar, so say that the lift could be smaller than it looks.")
print(verdict)

## Finish it

### Exercise 1: Finish it

**Task:** Do people who signed up in Europe buy Plus within 14 days more often than people in Asia Pacific? The counts and the pooled rate are given. Finish the test: the standard error of the gap under the null, z for Europe minus Asia Pacific, and the two-sided p-value.

**Expected:** `answer_1` is a list of two numbers, z rounded to 2 decimals and p to 3.

In [ ]:
europe = full[full["region"] == "europe"]["bought"]                  # given: 14-day conversion of people in Europe
asia = full[full["region"] == "asia_pacific"]["bought"]              # given: and in Asia Pacific
n_e, c_e = len(europe), int(europe.sum())
n_a, c_a = len(asia), int(asia.sum())
pooled_ea = (c_e + c_a) / (n_e + n_a)                                # given: the pooled rate
# the standard error of the gap under the null, then z for Europe minus Asia Pacific, then the two-sided p-value
answer_1 = None  # replace None with [z, p]: z rounded to 2 decimals, p to 3

In [ ]:
_ = check("Exercise 1", "answer_1", [1.41, 0.158], traps={(1.41, 0.079): "That is one tail. The p-value is two-sided: double the tail beyond |z|, as in unit 1."})

<details>
<summary>Hint</summary>

Unit 1: `se = np.sqrt(pooled * (1 - pooled) * (1 / n0 + 1 / n1))`, then `z = (p1 - p0) / se`, then `2 * stats.norm.sf(abs(z))`. Here the groups are Europe and Asia Pacific.

</details>

<details>
<summary>Solution</summary>

```python
se_ea = np.sqrt(pooled_ea * (1 - pooled_ea) * (1 / n_e + 1 / n_a))
z_ea = (c_e / n_e - c_a / n_a) / se_ea
answer_1 = [round(float(z_ea), 2), round(float(2 * stats.norm.sf(abs(z_ea))), 3)]
```

722 of the 5,638 people in Europe bought (12.8 percent) against 430 of the 3,639 in Asia Pacific (11.8 percent): z is 1.41 and the p-value 0.158, nowhere near 0.05. The 95 percent interval of the gap, 0.99 points, runs from -0.37 to 2.35, so a gap of 2 points is still possible. A one-tailed reading would give 0.079, and the check cell warns about it.

</details>

## On your own

### Exercise 2: On your own

**Task:** How much more active are people who signed up on the web than people who signed up on iOS in their first 14 days? Before you start, say in one sentence what the result looks like: a difference, its two interval ends and one standardized size. Run Welch's test on the activity of the two groups (people with a full 14 days of follow-up), and set `answer_2` to the list [difference of the means web minus iOS, low end, high end of its 95 percent interval, Cohen's d], each rounded to 2 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of four numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.99, 0.84, 1.14, 0.22], traps={(-0.99, -1.14, -0.84, -0.22): "These are the numbers for the other order of the groups, or for the wrong pair. Take the web people first and the iOS people second, and use equal_var=False."})

<details>
<summary>Hint</summary>

Unit 2 built the arrays `ios` and `android` from `full`. Do the same for `"web"`, call `stats.ttest_ind(web, ios, equal_var=False)`, ask the result for `.confidence_interval(confidence_level=0.95)`, and use `cohens_d` from unit 3.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A difference of a bit under one event, an interval a little either side of it, and a d between 0.2 and 0.4."*

```python
web = full.loc[full["platform"] == "web", "activity"].to_numpy()
res_w = stats.ttest_ind(web, ios, equal_var=False)
```

</details>

<details>
<summary>Solution</summary>

```python
web = full.loc[full["platform"] == "web", "activity"].to_numpy()
res_w = stats.ttest_ind(web, ios, equal_var=False)
ci_w = res_w.confidence_interval(confidence_level=0.95)
answer_2 = [round(float(web.mean() - ios.mean()), 2), round(float(ci_w.low), 2), round(float(ci_w.high), 2), round(float(cohens_d(web, ios)), 2)]
```

People who signed up on the web average 4.29 events against 3.30 on iOS: a difference of 0.99, interval 0.84 to 1.14, and d is 0.22, about twice the iOS and Android gap. The check cell reports the same numbers with the groups in the other order as a hint.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. The team planned to read paywall_v2 on its home market, `north_america`, on its own. A colleague says the lift there is not significant because the 95 percent intervals of the two groups' conversion rates overlap. Fix the line that sets `significant`, so that it follows from the test of the difference, with a bar of 0.05. Then `answer_3` is the list [p-value rounded to 3 decimals, whether the difference is significant].

**Expected:** `answer_3` is a list of a number and True or False.

In [ ]:
na = pw.merge(users[["user_id", "region"]], on="user_id")
na = na[na["region"] == "north_america"]                             # given: the home market, read on its own as planned before the test
n0_na, n1_na = int((na["variant"] == "control").sum()), int((na["variant"] == "variant").sum())
c0_na = int(na.loc[na["variant"] == "control", "converted"].sum())
c1_na = int(na.loc[na["variant"] == "variant", "converted"].sum())
pooled_na = (c0_na + c1_na) / (n0_na + n1_na)
z_na = (c1_na / n1_na - c0_na / n0_na) / np.sqrt(pooled_na * (1 - pooled_na) * (1 / n0_na + 1 / n1_na))
p_na = float(2 * stats.norm.sf(abs(z_na)))                           # given: the p-value of the difference


def interval(c, n):
    p = c / n
    s = np.sqrt(p * (1 - p) / n)
    return float(p - 1.96 * s), float(p + 1.96 * s)


(lo0, hi0), (lo1, hi1) = interval(c0_na, n0_na), interval(c1_na, n1_na)
significant = hi0 < lo1 or hi1 < lo0           # colleague: "significant when the two intervals do not overlap"
answer_3 = [round(p_na, 3), significant]

In [ ]:
_ = check("Exercise 3", "answer_3", [0.041, True], traps={(0.041, False): "The p-value is below 0.05, so the difference is significant. Whether two separate intervals overlap is not the test of the difference between them: decide with the p-value of the difference."})

<details>
<summary>Hint</summary>

Print `lo0, hi0, lo1, hi1` and `p_na`. The two intervals overlap, and the p-value of the difference is below 0.05. The test of a difference is the p-value of the difference, so the decision line should use `p_na`.

</details>

<details>
<summary>Solution</summary>

```python
significant = p_na < 0.05
answer_3 = [round(p_na, 3), significant]
```

In north_america, 143 of 1,129 control people bought (12.7 percent, interval 10.7 to 14.6) and 180 of 1,150 in the variant (15.7 percent, interval 13.5 to 17.8). The two intervals overlap by a point, yet the p-value of the difference is 0.041. The standard error of a difference, the square root of the sum of the two squared standard errors, is smaller than the sum of the two margins, so the test of the difference can find a gap that two overlapping intervals hide.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")